# Bronze — Central Bank of Iceland Ingestion

Fetches daily policy interest rates and monthly CPI from the Central Bank of Iceland XML API.

**Source:** Seðlabanki Íslands XML Time Series API  
**Groups:** GroupID=1 (policy rates), GroupID=3 (CPI)  
**Output:** `bronze.central_bank_raw` (Delta table)  
**Schedule:** Daily  

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from datetime import timedelta

BASE_URL = "https://sedlabanki.is/xmltimeseries/Default.aspx"
START_DATE = "2022-01-01"


def parse_central_bank_xml(xml_text):
    """Parse Seðlabanki XML response into a flat DataFrame."""
    root = ET.fromstring(xml_text)
    group_name = root.find("Name").text
    records = []

    for ts in root.findall("TimeSeries"):
        ts_id = ts.get("ID")
        ts_name = ts.find("Name").text

        for entry in ts.findall(".//Entry"):
            records.append({
                "group_name": group_name,
                "series_id":  ts_id,
                "series_name": ts_name,
                "date":  entry.find("Date").text,
                "value": float(entry.find("Value").text),
            })

    return pd.DataFrame(records)


if spark.catalog.tableExists("bronze.central_bank_raw"):
    last_date = spark.sql("SELECT MAX(date) FROM bronze.central_bank_raw").collect()[0][0]
    start_date = (last_date + timedelta(days=1)).strftime("%Y-%m-%d")
    print(f"Incremental load — fetching from {start_date}")
else:
    last_date = None
    start_date = START_DATE
    print(f"Full load — fetching from {start_date}")

In [ ]:
today = pd.Timestamp.now().strftime("%Y-%m-%d")

if start_date >= today:
    print(f"Table is already up to date (last date: {last_date}). Nothing to fetch.")
    new_data = None
else:
    # Policy rates (GroupID=1) — daily frequency
    resp_rates = requests.get(BASE_URL, params={"DagsFra": start_date, "GroupID": 1, "Type": "xml"})
    resp_rates.raise_for_status()
    df_rates = parse_central_bank_xml(resp_rates.text)
    df_rates["date"] = pd.to_datetime(df_rates["date"], format="%m/%d/%Y %I:%M:%S %p")
    df_rates["ingested_at"] = pd.Timestamp.now()
    print(f"Policy rates — {len(df_rates):,} rows | Series: {df_rates['series_name'].nunique()}")

    # CPI / inflation (GroupID=3) — monthly frequency
    resp_cpi = requests.get(BASE_URL, params={"DagsFra": start_date, "GroupID": 3, "Type": "xml"})
    resp_cpi.raise_for_status()
    df_cpi = parse_central_bank_xml(resp_cpi.text)
    df_cpi["date"] = pd.to_datetime(df_cpi["date"], format="%m/%d/%Y %I:%M:%S %p")
    df_cpi["ingested_at"] = pd.Timestamp.now()
    print(f"CPI — {len(df_cpi):,} rows | Series: {df_cpi['series_name'].unique()}")

    new_data = pd.concat([df_rates, df_cpi], ignore_index=True)

    if len(new_data) == 0:
        raise ValueError("[bronze_central_bank] Both APIs returned 0 rows - halting.")

In [ ]:
if new_data is not None:
    spark_df = spark.createDataFrame(new_data)
    spark_df.write.format("delta").mode("append").saveAsTable("bronze.central_bank_raw")
    print(f"Appended {spark_df.count():,} rows to bronze.central_bank_raw")
else:
    print("Nothing to write — table is already up to date.")